In [68]:
pdf_file_path = "grammar_books/kalamang.pdf"

In [69]:
import json
import pymupdf
import collections
import numpy as np
import re
from tqdm import tqdm


In [8]:
def extract_visual_lines(page):

    spans = []
    blocks = page.get_text("dict")["blocks"]

    for block in blocks:
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                sb = span["bbox"]
                spans.append({
                    "text": span["text"],
                    "x0": sb[0],
                    "y0": sb[1],
                    "x1": sb[2],
                    "y1": sb[3],
                    "font": span.get("font"),
                    "size": span.get("size"),
                    "block": block,  # optional
                })

    spans = sorted(spans, key=lambda s: s["y0"])
    if len(spans)==0:
        return None

    # Dynamic tolerance based on median height
    heights = [s["y1"] - s["y0"] for s in spans]
    median_h = np.median(heights) if heights else 10
    Y_TOL = median_h * 0.4

    visual_lines = []
    current_line = [spans[0]]

    for prev, curr in zip(spans, spans[1:]):
        if abs(curr["y0"] - prev["y0"]) < Y_TOL:
            current_line.append(curr)
        else:
            visual_lines.append(current_line)
            
            current_line = [curr]

    visual_lines.append(current_line)

    # Merge and sort spans within each line
    merged_lines = []
    for line_spans in visual_lines:
        sorted_spans = sorted(line_spans, key=lambda s: s["x0"])
        merged_text = " ".join([s["text"] for s in sorted_spans if s["text"].strip()])
        # Get most common font size and style by length weighting
        font_sizes = {}
        font_styles = {}
        for s in sorted_spans:
            length = len(s["text"])
            font_key = (s["font"], s["size"])
            font_sizes[font_key] = font_sizes.get(font_key, 0) + length
            font_styles[s["font"]] = font_styles.get(s["font"], 0) + length
        
        if font_sizes:
            common_font_size = max(font_sizes, key=font_sizes.get)
        else:
            common_font_size = (None, None)

        if font_styles:
            common_font_style = max(font_styles, key=font_styles.get)
        else:
            common_font_style = None

        
            
        
        merged_lines.append({
            "text": merged_text,
            "spans": sorted_spans,
            "x0": sorted_spans[0]["x0"],
            "y0": np.mean([s["y0"] for s in sorted_spans]),
            "x1": sorted_spans[-1]["x1"],
            "y1": max(s["y1"] for s in sorted_spans),
            "span_count": len(sorted_spans),
            "common_font_size": common_font_size[1],
            "common_font_style": common_font_style,
        })

    return merged_lines


In [71]:
# Load doc
print("Opening PDF file...")
doc = pymupdf.open(pdf_file_path)
print("PDF file opened successfully.")
print("Number of pages in PDF file:", doc.page_count)

Opening PDF file...
PDF file opened successfully.
Number of pages in PDF file: 573


In [72]:

font_size_counts = collections.Counter()
font_style_counts = collections.Counter()

In [73]:
all_merged_lines = []
for page in doc:
    page_num = page.number + 1
    
    merged_lines = extract_visual_lines(page)
    if merged_lines is None:
        continue

    for line in merged_lines:
        # Update font size and style counts
        font_size_counts.update([line["common_font_size"]])
        font_style_counts.update([line["common_font_style"]])
    
    all_merged_lines.append({
        "page_num": page_num,
        "lines": merged_lines
    })

    #for line in merged_lines:
        #print(line["text"], end="|")
    #print("---")
print("Most common font sizes:", font_size_counts.most_common(5))
print("Most common font styles:", font_style_counts.most_common(5))

Most common font sizes: [(10.909099578857422, 17600), (8.966400146484375, 674), (9.962599754333496, 243), (11.9552001953125, 139), (14.346199989318848, 98)]
Most common font styles: [('LibertinusSerif-Regular', 16226), ('LibertinusSerif-Italic', 2190), ('LibertinusSerif-Semibold', 529), ('LinuxLibertineG', 69), ('LibertinusMath-Regular', 27)]


In [74]:
most_common_font_size = font_size_counts.most_common(1)[0][0]
body_text_size = most_common_font_size
print("Determined body text size:", body_text_size)

Determined body text size: 10.909099578857422


In [87]:
# load doc
print("Opening PDF file...")
doc = pymupdf.open(pdf_file_path)   
print("PDF file opened successfully.")
print("Number of pages in PDF file:", doc.page_count)

# Render the pdf with the lines extracted with bounding boxes 

draw_lines = []
import re
odd = True
parallel_start = False
line_no = -1
count_from_start = 0

for row in all_merged_lines:
    page_num = row["page_num"]
    page = doc[page_num - 1]
    if page_num < 19 or (page_num > 80 and page_num < 135):
        continue

    for line in row["lines"]:
        if line["common_font_size"] < body_text_size - 0.1:
            continue
        
        line_no += 1
        # Quick parallel sentenece manual extraction
        bbox = (line["x0"], line["y0"], line["x1"], line["y1"])
        
        text = line["text"].strip().lower()

        # Check if line starts with (n) where n is a number
        if re.match(r"^\(\d+\)", text):
            # Get number inside ()
            number = int(re.findall(r"^\((\d+)\)", text)[0])
            if number > 999:
                continue
            #print(f"Page {page_num}: {line['text']}")
            #print(line["x0"], line["y0"], line["x1"], line["y1"])
            draw_lines.append({
                "page_num": page_num,
                "line_no": line_no,
                "text": line["text"],
                "line": line,
                "bbox": bbox,
                "start": True
            })
            #print("--1--")
            #print(line["common_font_size"], line["common_font_style"])
            parallel_start = True
            rect = pymupdf.Rect(bbox)
            color = (0, 1, 0)  # Green for start
            page.draw_rect(rect, color=color, width=0.5)
            count_from_start = 2
            continue

        # Roughly near 84 or 96 x value
        values = [84,88,90,96,100,108,114,120]
        if count_from_start > 0 and "LibertinusSerif" in line["common_font_style"]:
            #print("--2--")
            #print(line["common_font_size"], line["common_font_style"])
            draw_lines.append({
                "page_num": page_num,
                "line_no": line_no,
                "text": line["text"],
                "line": line,
                "bbox": bbox,
                "start": False
            })
            rect = pymupdf.Rect(bbox)
            color = (1, 0, 1)  # Magenta for continuation
            page.draw_rect(rect, color=color, width=0.5)
            count_from_start -= 1
            continue
        
        if any(abs(line["x0"] - v) < 3 for v in values) and "LibertinusSerif" in line["common_font_style"]:
            # print the line
            #print(f"Page {page_num}: {line['text']}")
            #print("--abs--")
            #print(line["common_font_size"], line["common_font_style"])
            draw_lines.append({
                "page_num": page_num,
                "line_no": line_no,
                "text": line["text"],
                "bbox": bbox,
                "line": line,
            })
            rect = pymupdf.Rect(bbox)
            color = (1, 0, 0) if odd else (0, 0, 1)
            page.draw_rect(rect, color=color, width=0.5)
            odd = not odd
        
        else:
            if parallel_start:
                #print("---")
                #print(f"Page {page_num}: {line['text']}")
                #print(line["x0"], line["y0"], line["x1"], line["y1"])
                pass
            parallel_start = False



doc.save("output_with_lines.pdf")

Opening PDF file...
PDF file opened successfully.
Number of pages in PDF file: 573


In [88]:
# Open the pdf again to draw blocks
print("Opening PDF file...")
doc = pymupdf.open(pdf_file_path)   
print("PDF file opened successfully.")

i = 0
cont_lines = 0
prev_line_no = -1
temp_block = []
final_blocks = []
while(i < len(draw_lines)-1):
    if i==0:
        temp_block.append(draw_lines[i])
        prev_line_no = draw_lines[i]["line_no"]
        i+=1
        continue
    curr_line = draw_lines[i]
    curr_line_no = curr_line["line_no"]
    if curr_line_no == prev_line_no + 1:
        temp_block.append(curr_line)
        prev_line_no = curr_line_no
        i+=1
    else:
        if len(temp_block) >0:
            print("Block:")
            # Check where the start block is
            start_indices = [idx for idx, bline in enumerate(temp_block) if bline.get("start")]
            page_numbers = [bline["page_num"] for bline in temp_block]
            #page_number_bool = [p in range(285, 476) for p in page_numbers]
            #print(page_number_bool)
            if start_indices:
                start_idx = start_indices[0]
                # Use start idx usually ignoring here for romansh as there are many without a (n) starting block
                final_blocks.append(temp_block[start_idx:])
                for bline in temp_block[start_idx:]:
                    print(f"Page {bline['page_num']}: {bline['text']}")
                    page_num = bline["page_num"]
                    page = doc[page_num - 1]
                    rect = pymupdf.Rect(bline["bbox"])
                    color = (0, 1, 0)  # Green for start
                    page.draw_rect(rect, color=color, width=0.5)
                print("-----")
        temp_block = [curr_line]
        prev_line_no = curr_line_no
        print(f"New block starting at line no {curr_line_no}")
        print(f"Current line text: {curr_line['text']}")
        
        i+=1
    
    
doc.save("output_with_lines_filtered.pdf")

Opening PDF file...
PDF file opened successfully.
Block:
New block starting at line no 54
Current line text: (1) hukat kon anggon yuwa
Block:
Page 20: (1) hukat kon anggon yuwa
Page 20: jaring_ikan satu milik_saya ini
Page 20: ‘jaring ikan milik saya satu ini’
-----
New block starting at line no 75
Current line text: (2) kaskas et-eir
Block:
Page 21: (2) kaskas et-eir
Page 21: burung_camar hidup-dua
Page 21: ‘dua burung camar’
-----
New block starting at line no 107
Current line text: (3) ma se guru
Block:
Page 22: (3) ma se guru
Page 22: dia sudah guru
Page 22: ‘Dia sudah (menjadi seorang) guru.’
Page 22: (4) ma me
Page 22: dia itu
Page 22: ‘Itu dia.’
Page 22: (5) im kansuor
Page 22: pisang empat
Page 22: ‘Ada empat pisang.’
-----
New block starting at line no 128
Current line text: (6) ka-mun narabir-in
Block:
Page 22: (6) ka-mun narabir-in
Page 22: kamu-jangan berteriak-jangan
Page 22: ‘Janganlah kamu berteriak!’
-----
New block starting at line no 150
Current line text: (7) an me w

In [38]:
final_blocks[0]

[{'page_num': 21,
  'line_no': 76,
  'text': '(1) I drùva schòn in téc da saprèndar anzjaman',
  'line': {'text': '(1) I drùva schòn in téc da saprèndar anzjaman',
   'spans': [{'text': '(1)',
     'x0': 64.20800018310547,
     'y0': 387.9992370605469,
     'x1': 74.63710021972656,
     'y1': 400.43560791015625,
     'font': 'LibertinusSerif-Regular',
     'size': 10.909099578857422,
     'block': {'number': 8,
      'type': 0,
      'bbox': (64.20800018310547,
       387.9992370605469,
       407.21246337890625,
       471.6976318359375),
      'lines': [{'spans': [{'size': 10.909099578857422,
          'flags': 4,
          'bidi': 0,
          'char_flags': 16,
          'font': 'LibertinusSerif-Regular',
          'color': 0,
          'alpha': 255,
          'ascender': 0.8939999938011169,
          'descender': -0.2460000067949295,
          'text': '(1)',
          'origin': (64.20800018310547, 397.7519836425781),
          'bbox': (64.20800018310547,
           387.999237060546

In [89]:
base_prompt_fine_segmentation = f"""
You are an expert linguist who analyzes text from descriptive grammars. 
Your task is to identify linguistic examples within a block of text and 
label their internal structure. You must be precise and must not hallucinate 
missing information. Only use what is provided.

You will receive the text as a sequence of numbered lines in the format:

[1] text of line 1
[2] text of line 2
...

DEFINITIONS:
A linguistic example begins with a line that starts with a parenthesized
example label such as (2), (15), (3a), (4b), etc.
All lines following this belong to the example until the next example label
line or the end of the block.

Inside each example, lines fall into four categories:
- SOURCE: The original language sentence, here tuatschin Romansh.
- GLOSS: A word-by-word gloss containing morphological markers (e.g., "-", "=", gloss abbreviations).
- TRANSLATION: A free translation line, generally inside single quotes.
- COMMENT: Any additional text that does not fit the above categories. Eg: Heading, normal text, table content. 

YOUR TASKS:
1. Identify each example in the block.
2. For each example, provide:
   - "example_label": the parenthesized label such as "(2)" or "(3a). (none) if no label is present".
   - "start_line": the line number where the example begins.
   - "end_line": the line number where the example ends.
3. For each line inside the example boundaries, assign a "role":
   one of "SOURCE", "GLOSS", "TRANSLATION", or "COMMENT".
4. If an example appears incomplete (missing gloss or translation), 
   include `"status": "incomplete"`. Otherwise mark `"status": "complete"`.
5. Return ONLY valid JSON. No explanations. No prose.

Below is the block of numbered lines you must analyze:

{{block_text}}

Return the final JSON only.
"""

In [90]:
print(len(final_blocks))

702


In [91]:

with open("openai_api_key.txt", "r") as f:
    openai_api_key = f.read().strip()

from openai import OpenAI
client = OpenAI(api_key=openai_api_key)


In [92]:
responses = []

for block in tqdm(final_blocks):
    block_text = ""
    for idx, line in enumerate(block):
        block_text += f"[{idx+1}] {line['text']}\n"
    
    #print(block_text)
    prompt = base_prompt_fine_segmentation.format(block_text=block_text)
    #print(prompt)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": ""},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=1500,
    )
    responses.append(response)
    #print("-----")

100%|██████████| 702/702 [36:51<00:00,  3.15s/it]


In [93]:
cleaned_responses = []

for response in responses:
    text = response.choices[0].message.content
    # Parse JSON
    # Remove ```json and ``` if present
    text = re.sub(r"^```json", "", text.strip())
    text = re.sub(r"```$", "", text.strip())
    data = json.loads(text)
    print(len(data))
    #print(data)
    cleaned_responses.append(data)

1
1
3
1
2
3
1
1
1
3
1
1
3
3
5
5
1
1
1
4
2
3
4
2
3
1
2
1
4
3
2
1
1
1
1
1
1
1
1
1
1
2
2
2
1
2
2
1
4
2
2
2
4
1
3
2
4
2
1
2
1
1
1
1
4
1
1
1
1
1
1
2
4
2
1
1
2
2
1
1
1
1
1
1
1
1
2
1
5
2
1
1
1
2
1
1
1
2
4
1
2
1
2
2
1
1
1
1
1
3
3
2
1
2
1
1
1
1
2
2
1
2
1
1
1
1
0
3
2
1
2
4
1
5
1
2
1
2
1
1
2
1
2
1
2
1
2
1
1
4
1
2
1
1
3
2
1
1
2
1
1
2
1
2
2
2
2
1
3
2
3
1
1
1
1
1
3
2
1
2
2
2
1
0
2
1
1
1
1
1
1
1
1
0
2
1
1
1
1
1
1
1
2
3
3
1
1
1
2
2
2
2
1
2
4
1
2
1
2
2
1
1
2
1
3
1
2
2
1
2
2
1
1
4
2
2
2
1
1
2
4
2
2
1
1
1
1
2
1
1
1
1
1
3
1
1
2
1
1
3
3
1
1
1
0
2
2
3
1
1
2
2
1
1
2
1
1
1
1
3
2
0
2
2
1
2
1
1
2
1
0
1
2
1
1
1
1
2
1
1
1
1
2
2
3
3
1
2
1
1
2
3
4
5
1
2
5
3
1
1
1
1
3
2
1
2
2
2
2
1
1
2
2
1
2
1
3
1
2
1
2
1
1
1
3
3
1
3
3
2
1
2
2
1
3
2
1
1
1
2
1
1
1
2
2
1
1
2
2
1
2
3
2
1
2
4
1
1
2
1
1
1
1
1
2
1
2
1
1
1
1
1
1
2
3
1
1
1
1
1
2
1
2
1
1
2
2
1
1
2
1
1
1
1
1
5
2
2
1
1
2
2
3
1
1
2
2
1
1
3
2
1
1
1
4
1
1
2
2
1
4
1
1
2
2
1
1
2
1
1
3
1
1
2
1
1
1
1
2
2
2
4
1
1
2
1
1
3
1
3
3
1
1
1
1
1
2
8
1
1
1
2
6
3
2
2
1
1
4
1
1
1
1
5
0
1
1
2
4
1


In [94]:
# Save cleaned responses
with open("kalamang_cleaned_responses_parallel_new.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_responses, f, indent=2, ensure_ascii=False)

In [34]:
with open("romansh_cleaned_responses_parallel.json", "r", encoding="utf-8") as f:
    cleaned_responses = json.load(f)

In [95]:
responses_flattened = []
for block in cleaned_responses:
    if "examples" in block:
        block = block["examples"]
    for example in block:
        if type(example) is str:
            print("Skipping invalid example:", example)
            print(block)
            continue
        for line in example["lines"]:
            #print(line)
            line_no = line['line_number'] if 'line_number' in line else 'N/A'
            if line_no == 'N/A':
                line_no = line['line'] if 'line' in line else None
                
            print(f"{line_no}: {line['role']}")
            if line["role"] == "TRANSLATION_COMMENT":
                line["role"] = "TRANSLATION"
            responses_flattened.append({
                "example_label": example["example_label"],
                "line_number": line_no,
                "role": line["role"],
            })


1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
7: SOURCE
8: GLOSS
9: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
7: SOURCE
8: GLOSS
9: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: GLOSS
4: SOURCE
5: GLOSS
6: GLOSS
7: TRANSLATION
8: COMMENT
1: SOURCE
2: SOURCE
3: SOURCE
4: SOURCE
5: SOURCE
6: SOURCE
7: SOURCE
8: SOURCE
1: COMMENT
1: SOURCE
2: GLOSS
3: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
7: SOURCE
8: GLOSS
9: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
7: SOURCE
8: GLOSS
9: TRANSLATION
1: SOURCE
2: GLOSS
3: TRANSLATION
4: SOURCE
5: GLOSS
6: TRANSLATION
7: SOURCE
8: GLOSS
9: TRANSLATION
10: SOURCE
11: GLOSS
12: TRANSLATION
13: SOUR

In [96]:
# Draw the extracted examples on the pdf
print("Opening PDF file...")
doc = pymupdf.open(pdf_file_path)
print("PDF file opened successfully.")


temp_role = None
prev_line = -1
final_roles = []

prev_role = None


for block, resp in zip(final_blocks, cleaned_responses):
    if "examples" in resp:  
        resp = resp["examples"]
    for example in resp:
        start_line_idx = example["start_line"] - 1
        end_line_idx = example["end_line"] - 1
        for line_idx in range(start_line_idx, end_line_idx + 1):
            actual_role = None
            bline = block[line_idx]
            page_num = bline["page_num"]
            page = doc[page_num - 1]
            bbox = bline["bbox"]
            rect = pymupdf.Rect(bbox)
            role = example["lines"][line_idx - start_line_idx]["role"]
            if role == "SOURCE":
                color = (0, 0, 1)  # Blue
            elif role == "GLOSS":
                color = (0, 1, 0)  # Green
            elif role == "TRANSLATION":
                color = (1, 0, 0)  # Red
            else:
                color = (1, 1, 0)  # Yellow for COMMENT

            
            if role == "GLOSS":
                # Check if line has italics in one if it's fonts
                spans = bline["line"]["spans"]
                #print("Line:", bline["text"])
                #print([s["font"] for s in spans])
                has_italics = any("Italic" in s["font"] for s in spans) or any("Semibold" in s["font"] for s in spans)
                if has_italics:
                    color = (0, 0, 0)  # Black for GLOSS with italics
                    actual_role = "SOURCE"
                    
            
            elif temp_role is not None and role == "COMMENT" and line_idx == prev_line + 1:
                color = (1, 0.5, 0)  # Orange for COMMENT after other roles
                actual_role = "TRANSLATION"
                prev_line = line_idx

            # Check if the gloss

            else:
                temp_role = None
            if temp_role is None and role != "SOURCE" and role!="COMMENT":
                temp_role = role
                prev_line = line_idx

            
            final_roles.append({
                "page_num": page_num,
                "bline": bline,
                "role": actual_role if actual_role is not None else role,
                
            })
            print(f"Page {page_num}, Line: {bline['text']}, Role: {actual_role if actual_role is not None else role}")
            
            

            page.draw_rect(rect, color=color, width=0.5)
doc.save("output_with_fine_segmentation.pdf")

Opening PDF file...
PDF file opened successfully.
Page 20, Line: (1) hukat kon anggon yuwa, Role: SOURCE
Page 20, Line: jaring_ikan satu milik_saya ini, Role: GLOSS
Page 20, Line: ‘jaring ikan milik saya satu ini’, Role: TRANSLATION
Page 21, Line: (2) kaskas et-eir, Role: SOURCE
Page 21, Line: burung_camar hidup-dua, Role: GLOSS
Page 21, Line: ‘dua burung camar’, Role: TRANSLATION
Page 22, Line: (3) ma se guru, Role: SOURCE
Page 22, Line: dia sudah guru, Role: GLOSS
Page 22, Line: ‘Dia sudah (menjadi seorang) guru.’, Role: TRANSLATION
Page 22, Line: (4) ma me, Role: SOURCE
Page 22, Line: dia itu, Role: GLOSS
Page 22, Line: ‘Itu dia.’, Role: TRANSLATION
Page 22, Line: (5) im kansuor, Role: SOURCE
Page 22, Line: pisang empat, Role: GLOSS
Page 22, Line: ‘Ada empat pisang.’, Role: TRANSLATION
Page 22, Line: (6) ka-mun narabir-in, Role: SOURCE
Page 22, Line: kamu-jangan berteriak-jangan, Role: GLOSS
Page 22, Line: ‘Janganlah kamu berteriak!’, Role: TRANSLATION
Page 23, Line: (7) an me watko

In [97]:
for row in final_roles:
    print(f"Page {row['page_num']}: {row['bline']['text']} --> {row['role']}")

Page 20: (1) hukat kon anggon yuwa --> SOURCE
Page 20: jaring_ikan satu milik_saya ini --> GLOSS
Page 20: ‘jaring ikan milik saya satu ini’ --> TRANSLATION
Page 21: (2) kaskas et-eir --> SOURCE
Page 21: burung_camar hidup-dua --> GLOSS
Page 21: ‘dua burung camar’ --> TRANSLATION
Page 22: (3) ma se guru --> SOURCE
Page 22: dia sudah guru --> GLOSS
Page 22: ‘Dia sudah (menjadi seorang) guru.’ --> TRANSLATION
Page 22: (4) ma me --> SOURCE
Page 22: dia itu --> GLOSS
Page 22: ‘Itu dia.’ --> TRANSLATION
Page 22: (5) im kansuor --> SOURCE
Page 22: pisang empat --> GLOSS
Page 22: ‘Ada empat pisang.’ --> TRANSLATION
Page 22: (6) ka-mun narabir-in --> SOURCE
Page 22: kamu-jangan berteriak-jangan --> GLOSS
Page 22: ‘Janganlah kamu berteriak!’ --> TRANSLATION
Page 23: (7) an me watko nawanggar --> SOURCE
Page 23: saya topik disini tunggu --> GLOSS
Page 23: ‘Untuk saya, saya menunggu di sini.’ --> TRANSLATION
Page 23: (8) an-a watko mu-a metko --> SOURCE
Page 23: saya-fokus disini mereka-fokus disa

In [98]:
missing_parrallel_lines = []
page_nos = [188,229,278,452]
for page in all_merged_lines:
    page_no = page["page_num"]
    if page_no not in page_nos:
        continue
    print("--- Page", page_no, "---")
    for line_no,line in enumerate(page["lines"]):
        x0 = line["x0"]
        text = line["text"]
        bbox = (line["x0"], line["y0"], line["x1"], line["y1"])
        line["bbox"] = bbox
        #print(text,x0)
        #print(text, x0)
        values = [99,113,142,85]
        if any(abs(x0 - v) < 5 for v in values) and "LibertinusSerif" in line["common_font_style"]:
            # Check if this line is in final_roles
            found = False
            for row in final_roles:
                if row["page_num"] == page_no and row["bline"]["text"] == text:
                    found = True
                    #print("FOUND PARALLEL LINE:")
                    print(f" --- Page {page_no}: {text} (x0={x0})")
                    break
            if not found:
                missing_parrallel_lines.append({
                    "page_num": page_no,
                    "bline": {
                        "page_num": page_no,
                        "line_no": line_no,
                        "text": text,
                        "bbox": bbox,
                        "line": line,
                        "start": False
                    },
                    
                })
                #print("MISSING PARALLEL LINE!")
                print(f"Page {page_no}: {text} (x0={x0})")

--- Page 188 ---
Page 188: c.  kor kasir (x0=99.95100402832031)
Page 188: leg joint (x0=114.04000091552734)
Page 188: ‘ankle’ (x0=113.16699981689453)
Page 188: d.  kor-pak (x0=99.0999984741211)
Page 188: leg-moon (x0=114.04000091552734)
Page 188: ‘knee’ (x0=113.16699981689453)
Page 188: For many right-headed compounds, the choice between using or not using  -un (x0=80.35800170898438)
 --- Page 188: a.  kokok-nar (x0=99.63400268554688)
 --- Page 188: chicken-egg (x0=114.04000091552734)
 --- Page 188: ‘chicken egg’ (x0=113.16699981689453)
 --- Page 188: b.  kokok nar-un (x0=99.45999908447266)
 --- Page 188: chicken egg (x0=114.04000091552734)
 --- Page 188: ‘chicken egg’ (x0=113.16699981689453)
 --- Page 188: b.  lempuang  ‘island’ → lempuangpuang  ‘islands’ (x0=99.45999908447266)
 --- Page 188: c.  tumun  ‘child’ → tumtum  ‘children’ (x0=99.95100402832031)
--- Page 229 ---
Page 229: 57 purap talinramandalin (x0=85.5469970703125)
Page 229: 98 putkaninggonie talinirie (x0=85.5469970703125

In [99]:
manual_roles = "SGTSGTCCCSTSTSTTSTSTSTSTSGTCCCCCCCCCCTC"
manual_roles = [x for x in manual_roles]
mapping = {
    "S": "SOURCE",
    "G": "GLOSS",
    "T": "TRANSLATION",
    "C": "COMMENT"
}
manual_missing_parallel_lines_classification = [mapping[x] for x in manual_roles]

In [100]:
len(manual_missing_parallel_lines_classification), len(missing_parrallel_lines)

(39, 39)

In [101]:
missing_parrallel_lines_filtered = []

for i,row in enumerate(missing_parrallel_lines):

    print(f"Page {row['page_num']}: {row['bline']['text']} ---> {manual_missing_parallel_lines_classification[i]}")
    if manual_missing_parallel_lines_classification[i] != "COMMENT":
        row["role"] = manual_missing_parallel_lines_classification[i]
        missing_parrallel_lines_filtered.append(row)

Page 188: c.  kor kasir ---> SOURCE
Page 188: leg joint ---> GLOSS
Page 188: ‘ankle’ ---> TRANSLATION
Page 188: d.  kor-pak ---> SOURCE
Page 188: leg-moon ---> GLOSS
Page 188: ‘knee’ ---> TRANSLATION
Page 188: For many right-headed compounds, the choice between using or not using  -un ---> COMMENT
Page 229: 57 purap talinramandalin ---> COMMENT
Page 229: 98 putkaninggonie talinirie ---> COMMENT
Page 229: ‘two thousand four hundred fifty and six’ ---> SOURCE
Page 229: 8721 ripirie reitramandalin purir ba kon ---> TRANSLATION
Page 229: ‘eight thousand seven hundred twenty and one’ ---> SOURCE
Page 229: 72,568 ripi putramandalin talinir reirap putraman talinirie ---> TRANSLATION
Page 229: ‘seventy and two thousand five hundred sixty and eight’ ---> SOURCE
Page 229: 526,389 ripi reirap purir ba raman reitkaruok putirie talinkaninggonie ---> TRANSLATION
Page 229: ‘five hundred and twenty and six thousand three hundred eighty ---> TRANSLATION
Page 229: and nine’ ---> SOURCE
Page 229: 1,500,0

In [102]:
missing_parrallel_lines_filtered


[{'page_num': 188,
  'bline': {'page_num': 188,
   'line_no': 1,
   'text': 'c.  kor kasir',
   'bbox': (99.95100402832031,
    np.float64(78.87025451660156),
    152.2000274658203,
    91.30663299560547),
   'line': {'text': 'c.  kor kasir',
    'spans': [{'text': 'c.',
      'x0': 99.95100402832031,
      'y0': 78.87025451660156,
      'x1': 107.02009582519531,
      'y1': 91.30663299560547,
      'font': 'LibertinusSerif-Regular',
      'size': 10.909099578857422,
      'block': {'number': 1,
       'type': 0,
       'bbox': (99.0999984741211,
        78.87025451660156,
        156.53094482421875,
        165.2106475830078),
       'lines': [{'spans': [{'size': 10.909099578857422,
           'flags': 4,
           'bidi': 0,
           'char_flags': 16,
           'font': 'LibertinusSerif-Regular',
           'color': 0,
           'alpha': 255,
           'ascender': 0.8939999938011169,
           'descender': -0.2460000067949295,
           'text': 'c.',
           'origin': (99.9

In [103]:
final_roles[0]

{'page_num': 20,
 'bline': {'page_num': 20,
  'line_no': 54,
  'text': '(1) hukat kon anggon yuwa',
  'line': {'text': '(1) hukat kon anggon yuwa',
   'spans': [{'text': '(1)',
     'x0': 75.5469970703125,
     'y0': 439.1192626953125,
     'x1': 85.97610473632812,
     'y1': 451.5556335449219,
     'font': 'LibertinusSerif-Regular',
     'size': 10.909099578857422,
     'block': {'number': 9,
      'type': 0,
      'bbox': (75.5469970703125,
       439.1192626953125,
       246.74319458007812,
       480.48760986328125),
      'lines': [{'spans': [{'size': 10.909099578857422,
          'flags': 4,
          'bidi': 0,
          'char_flags': 16,
          'font': 'LibertinusSerif-Regular',
          'color': 0,
          'alpha': 255,
          'ascender': 0.8939999938011169,
          'descender': -0.2460000067949295,
          'text': '(1)',
          'origin': (75.5469970703125, 448.87200927734375),
          'bbox': (75.5469970703125,
           439.1192626953125,
           85.97

In [104]:
combined_final_roles = final_roles + missing_parrallel_lines_filtered

In [105]:
len(combined_final_roles)

4322

In [106]:
# Save file
with open("kalamang_combined_final_roles.json","w",encoding="utf-8") as f:
    json.dump(combined_final_roles,f)

In [253]:
print(combined_final_roles[0].keys())
print(combined_final_roles[0]["bline"].keys())
print(combined_final_roles[0]["bline"]["line"].keys())

dict_keys(['page_num', 'bline', 'role'])
dict_keys(['page_num', 'line_no', 'text', 'line', 'bbox', 'start'])
dict_keys(['text', 'spans', 'x0', 'y0', 'x1', 'y1', 'span_count', 'common_font_size', 'common_font_style'])


In [254]:
print(missing_parrallel_lines_filtered[0].keys())
print(missing_parrallel_lines_filtered[0]["bline"].keys())

dict_keys(['page_num', 'bline', 'role'])
dict_keys(['page_num', 'line_no', 'text', 'bbox', 'line', 'start'])


In [281]:
prev_roles = []
prev_line_no = -1000
prev_page_no = -1
filter_indices = []
block_indices = []


i = 0
while i < len(combined_final_roles):
    row = combined_final_roles[i]
    line_no = row["bline"]["line_no"] if "line_no" in row["bline"] else None
    page_no = row["page_num"]
    if line_no == prev_line_no + 1:
        prev_roles.append(row["role"])
        prev_line_no = line_no
        prev_page_no = page_no
        block_indices.append(i)
        
    else:
        # new block. Check the last block 
        if prev_roles!=[]: 
            #print(prev_page_no,prev_roles)
            if all(r == "SOURCE" or r=="COMMENT" for r in prev_roles):
                print("FILTERING BLOCK INDICES:", block_indices)
                filter_indices.extend(block_indices)
            elif "GLOSS" in prev_roles:
                double_gloss_found = False
                # Check for double GLOSS appearing consecutively
                for j in range(1,len(prev_roles)):
                    if prev_roles[j]=="GLOSS" and prev_roles[j-1]=="GLOSS":
                        print("DOUBLE GLOSS")
                        prev_roles[j-1] = "SOURCE"
                        double_gloss_found = True
                        #filter_indices.extend(block_indices)
                        #break
                   
                # Change the roles in combined_final_roles
                if double_gloss_found:
                    print(prev_roles)
                    for j,block_idx in enumerate(block_indices):
                        combined_final_roles[block_idx]["role"] = prev_roles[j]
                        print(f"Changing role at index {block_idx} to {prev_roles[j]}")
            elif "TRANSLATION" in prev_roles and "GLOSS" not in prev_roles and prev_page_no not in [443,446]:
                

                # Check for double source and change one to gloss
                double_source_found = False
                print(prev_page_no,prev_roles)
                for j in range(1,len(prev_roles)):
                    if prev_roles[j]=="SOURCE" and prev_roles[j-1]=="SOURCE":
                        print("DOUBLE SOURCE")
                        prev_roles[j] = "GLOSS"
                        double_source_found = True
                        #filter_indices.extend(block_indices)
                        #break
                if double_source_found:
                    print(prev_roles)
                    for j,block_idx in enumerate(block_indices):
                        
                        combined_final_roles[block_idx]["role"] = prev_roles[j]
                        #print(f"Changing role at index {block_idx} to {prev_roles[j]}")

        block_indices = []
        prev_line_no = line_no
        prev_page_no = page_no
        prev_roles = [row["role"]]
        block_indices.append(i)

    #print(f"Page {page_no}, Line {line_no}: {row['bline']['text']} --> {row['role']}")
    
    i+=1

22 ['SOURCE', 'SOURCE', 'TRANSLATION', 'SOURCE', 'SOURCE', 'TRANSLATION', 'SOURCE', 'SOURCE', 'TRANSLATION']
DOUBLE SOURCE
DOUBLE SOURCE
DOUBLE SOURCE
['SOURCE', 'GLOSS', 'TRANSLATION', 'SOURCE', 'GLOSS', 'TRANSLATION', 'SOURCE', 'GLOSS', 'TRANSLATION']
FILTERING BLOCK INDICES: [47, 48, 49, 50, 51, 52, 53, 54]
FILTERING BLOCK INDICES: [55]
FILTERING BLOCK INDICES: [132, 133, 134, 135]
FILTERING BLOCK INDICES: [238, 239, 240]
FILTERING BLOCK INDICES: [241, 242, 243]
FILTERING BLOCK INDICES: [244]
FILTERING BLOCK INDICES: [273, 274, 275, 276, 277, 278, 279, 280]
FILTERING BLOCK INDICES: [547, 548, 549, 550, 551]
179 ['SOURCE', 'SOURCE', 'TRANSLATION']
DOUBLE SOURCE
['SOURCE', 'GLOSS', 'TRANSLATION']
FILTERING BLOCK INDICES: [591, 592, 593, 594, 595, 596]
183 ['SOURCE', 'TRANSLATION', 'SOURCE', 'TRANSLATION', 'SOURCE', 'TRANSLATION', 'SOURCE', 'TRANSLATION']
FILTERING BLOCK INDICES: [619, 620]
FILTERING BLOCK INDICES: [691, 692, 693]
FILTERING BLOCK INDICES: [703, 704]
FILTERING BLOCK IND

In [284]:
# Render the final combined roles on the pdf
print("Opening PDF file...")
doc = pymupdf.open(pdf_file_path)
print("PDF file opened successfully.")
# Render the pdf with the lines extracted with bounding boxes

for i,row in enumerate(combined_final_roles):
    if i in filter_indices:
        continue

    text = row["bline"]["text"].strip()
    if "to chew betel" in text.lower():
        print("Skipped line with 'to chew betel'")
        continue
    page_num = row["page_num"]
    page = doc[page_num - 1]
    bbox = row["bline"]["bbox"]
    rect = pymupdf.Rect(bbox)
    role = row["role"]
    if role == "SOURCE":
        color = (0, 0, 1)  # Blue
    elif role == "GLOSS":
        color = (0, 1, 0)  # Green
    elif role == "TRANSLATION":
        color = (1, 0, 0)  # Red
    else:
        color = (1, 1, 0)  # Yellow for COMMENT

    page.draw_rect(rect, color=color, width=0.5)

doc.save("output_with_combined_final_roles.pdf")
doc.close()

Opening PDF file...
PDF file opened successfully.
Skipped line with 'to chew betel'


In [285]:
combined_final_roles_filtered = [row for i,row in enumerate(combined_final_roles) if i not in filter_indices ]
print(len(combined_final_roles))
combined_final_roles_filtered = [row for row in combined_final_roles_filtered if "to chew betel" not in row["bline"]["text"].lower()]
print(len(combined_final_roles_filtered))
combined_final_roles_filtered = [row for row in combined_final_roles_filtered if row["role"]!="COMMENT"]
print(len(combined_final_roles_filtered))

4355
4243
4224


In [286]:
# Store final filtered roles
with open("combined_final_roles_filtered.json","w",encoding="utf-8") as f:
    json.dump(combined_final_roles_filtered,f)

# Parallel sentence classifier

In [39]:
import json
import pandas as pd

def load_and_flatten(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        rows = json.load(f)

    data = []
    for row in rows:
        b = row["bline"]
        data.append({
            "page": row["page_num"],
            "line_no": b["line_no"],
            "text": b["text"],
            "x0": b["bbox"][0],
            "y0": b["bbox"][1],
            "x1": b["bbox"][2],
            "y1": b["bbox"][3],
            "width": b["bbox"][2] - b["bbox"][0],
            "height": b["bbox"][3] - b["bbox"][1],
            "role": row["role"]
        })
         

    return data

curated_data = load_and_flatten("combined_final_roles_filtered.json")
curated_lookup = {}
for row in curated_data:

    curated_lookup[(row["page"], row["text"])] = row["role"]



In [40]:
print(len(curated_lookup))
print(len(curated_data))

4213
4224


In [41]:
# Load the entire line table with role as OTHER
all_data = []
count = 0
all_lines_count = 0
for row in all_merged_lines:
    
    lines = row["lines"]
    for line_no,line in enumerate(lines):
        #print(line.keys())
        all_lines_count += 1
        role = "OTHER"
        curated_row = curated_lookup.get((row["page_num"], line["text"]))
        if curated_row is not None:
            role = curated_row
            
            count += 1
            
        all_data.append({
            "page": row["page_num"],
            "line_no": line_no,
            "text": line["text"],
            "x0": line["x0"],
            "y0": line["y0"],
            "x1": line["x1"],
            "y1": line["y1"],
            "width": line["x1"] - line["x0"],
            "height": line["y1"] - line["y0"],
            "role": role
        })

print("Number of curated lines skipped:", count)
print("Total lines processed:", all_lines_count)
print("Length of curated data:", len(curated_data))

Number of curated lines skipped: 0
Total lines processed: 20150
Length of curated data: 4224


In [42]:
print(len(all_data))

20150


In [43]:
df = pd.DataFrame(all_data)
df.head()

,page,line_no,text,x0,y0,x1,y1,width,height,role
0,1,0,A grammar of,46.354000,50.835377,351.912201,100.382675,305.558201,49.547298,OTHER
1,1,1,Tuatschin,44.818001,99.733368,257.772308,149.280670,212.954308,49.547302,OTHER
2,1,2,A Sursilvan Romansh dialect,46.354000,167.793213,354.761932,191.613998,308.407932,23.820786,OTHER
3,1,3,Philippe Maurer-Cecchini,46.354000,221.051514,311.559509,249.445038,265.205509,28.393524,OTHER
4,1,4,Comprehensive Grammar Library 3 language science,46.354000,592.548401,431.578400,606.221252,385.224400,13.672852,OTHER


In [339]:
df["center_x"] = (df["x0"] + df["x1"]) / 2
df["center_y"] = (df["y0"] + df["y1"]) / 2
df["area"] = df["width"] * df["height"]
df["aspect_ratio"] = df["width"] / (df["height"] + 1e-6)
df.head()


,page,line_no,text,x0,y0,x1,y1,width,height,role,center_x,center_y,area,aspect_ratio
0,1,0,A grammar of,46.354000,50.835377,351.912201,100.382675,305.558201,49.547298,OTHER,199.133101,75.609026,15139.583365,6.167000
1,1,1,Kalamang,46.354000,99.733368,269.415955,149.280670,223.061954,49.547302,OTHER,157.884977,124.507019,11052.118079,4.502000
2,1,2,Eline Visser,46.354000,183.008514,169.292999,211.402039,122.938999,28.393524,OTHER,107.823500,197.205276,3490.671445,4.329825
3,1,3,Comprehensive Grammar Library 4 language science,46.354000,592.548401,431.578400,606.221252,385.224400,13.672852,OTHER,238.966200,599.384827,5267.116034,28.174399
4,1,4,press,403.896912,606.764160,423.339966,614.754333,19.443054,7.990173,OTHER,413.618439,610.759247,155.353373,2.433370


In [343]:
from sklearn.model_selection import train_test_split

# === Your split ===
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["role"]
)

print("Training set size:", len(train_df))
print("Test set size:", len(test_df))


Training set size: 16231
Test set size: 2865


In [346]:
from catboost import CatBoostClassifier, Pool


# === Features ===
text_features = ["text"]
num_features = [
    "page","line_no","x0","y0","x1","y1",
    "width","height","center_x","center_y",
    "area","aspect_ratio"
]

feature_cols = num_features + text_features

# CatBoost expects column indices for text features
text_feature_indices = [feature_cols.index("text")]

# === Pools ===
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df["role"],
    text_features=text_feature_indices
)

test_pool = Pool(
    data=test_df[feature_cols],
    label=test_df["role"],
    text_features=text_feature_indices
)

# === Model ===
model = CatBoostClassifier(
    iterations=100,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    eval_metric="TotalF1",
    task_type="CPU"
)

model.fit(train_pool, eval_set=test_pool, verbose=50)


0:	learn: 0.9236584	test: 0.9363879	best: 0.9363879 (0)	total: 877ms	remaining: 1m 26s
50:	learn: 0.9680546	test: 0.9700091	best: 0.9700091 (48)	total: 46.1s	remaining: 44.3s
99:	learn: 0.9717044	test: 0.9722861	best: 0.9730120 (77)	total: 1m 30s	remaining: 0us

bestTest = 0.9730120359
bestIteration = 77

Shrink model to first 78 iterations.


In [347]:
# Store the model
model.save_model("kalamang_line_role_classifier.cbm")


In [348]:
# Read different grammar book
new_pdf_file_path = "grammar_books/sursilvan_romansh.pdf"

# Load doc
print("Opening new PDF file...")
new_doc = pymupdf.open(new_pdf_file_path)
print("New PDF file opened successfully.")

Opening new PDF file...
New PDF file opened successfully.


In [349]:
all_merged_lines_romansh = []
for page in new_doc:
    page_num = page.number + 1
    
    merged_lines = extract_visual_lines(page)
    if merged_lines is None:
        continue
    all_merged_lines_romansh.append({
        "page_num": page_num,
        "lines": merged_lines
    })



In [350]:
# prepare data for prediction
all_data_romansh = []
for row in all_merged_lines_romansh:
    lines = row["lines"]
    for line_no,line in enumerate(lines):
        all_data_romansh.append({
            "page": row["page_num"],
            "line_no": line_no,
            "text": line["text"],
            "x0": line["x0"],
            "y0": line["y0"],
            "x1": line["x1"],
            "y1": line["y1"],
            "width": line["x1"] - line["x0"],
            "height": line["y1"] - line["y0"],
            "role": "OTHER"  # placeholder
        })
df_romansh = pd.DataFrame(all_data_romansh)
df_romansh["center_x"] = (df_romansh["x0"] + df_romansh["x1"]) / 2
df_romansh["center_y"] = (df_romansh["y0"] + df_romansh["y1"]) / 2
df_romansh["area"] = df_romansh["width"] * df_romansh["height"]
df_romansh["aspect_ratio"] = df_romansh["width"] / (df_romansh["height"] + 1e-6)
df_romansh.head()

,page,line_no,text,x0,y0,x1,y1,width,height,role,center_x,center_y,area,aspect_ratio
0,1,0,A grammar of,46.354000,50.835377,351.912201,100.382675,305.558201,49.547298,OTHER,199.133101,75.609026,15139.583365,6.167000
1,1,1,Tuatschin,44.818001,99.733368,257.772308,149.280670,212.954308,49.547302,OTHER,151.295155,124.507019,10551.311441,4.298000
2,1,2,A Sursilvan Romansh dialect,46.354000,167.793213,354.761932,191.613998,308.407932,23.820786,OTHER,200.557966,179.703606,7346.519208,12.947009
3,1,3,Philippe Maurer-Cecchini,46.354000,221.051514,311.559509,249.445038,265.205509,28.393524,OTHER,178.956755,235.248276,7530.119035,9.340352
4,1,4,Comprehensive Grammar Library 3 language science,46.354000,592.548401,431.578400,606.221252,385.224400,13.672852,OTHER,238.966200,599.384827,5267.116034,28.174399


In [355]:
# Run inference
preds = model.predict(df_romansh[feature_cols])
probs = model.predict_proba(df_romansh[feature_cols])
print(preds)

[['OTHER']
 ['OTHER']
 ['OTHER']
 ...
 ['OTHER']
 ['OTHER']
 ['OTHER']]


In [356]:
df_romansh["predicted_role"] = preds.ravel()
df_romansh["prediction_confidence"] = probs.max(axis=1)
df_romansh.head()

,page,line_no,text,x0,y0,x1,y1,width,height,role,center_x,center_y,area,aspect_ratio,predicted_role,prediction_confidence
0,1,0,A grammar of,46.354000,50.835377,351.912201,100.382675,305.558201,49.547298,OTHER,199.133101,75.609026,15139.583365,6.167000,OTHER,0.974587
1,1,1,Tuatschin,44.818001,99.733368,257.772308,149.280670,212.954308,49.547302,OTHER,151.295155,124.507019,10551.311441,4.298000,OTHER,0.967829
2,1,2,A Sursilvan Romansh dialect,46.354000,167.793213,354.761932,191.613998,308.407932,23.820786,OTHER,200.557966,179.703606,7346.519208,12.947009,OTHER,0.974936
3,1,3,Philippe Maurer-Cecchini,46.354000,221.051514,311.559509,249.445038,265.205509,28.393524,OTHER,178.956755,235.248276,7530.119035,9.340352,OTHER,0.969545
4,1,4,Comprehensive Grammar Library 3 language science,46.354000,592.548401,431.578400,606.221252,385.224400,13.672852,OTHER,238.966200,599.384827,5267.116034,28.174399,OTHER,0.980075


In [363]:
import fitz

def render_predictions_with_labels(pdf_path, df_pred, output_path):
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        subset = df_pred[df_pred.page == page_num + 1]

        for _, row in subset.iterrows():
            bbox = fitz.Rect(row["x0"], row["y0"], row["x1"], row["y1"])

            # Colors
            role = row["predicted_role"]
            if role == "SOURCE":
                color = (0, 0, 1)
            elif role == "GLOSS":
                color = (0, 1, 0)
            elif role == "TRANSLATION":
                color = (1, 0, 0)
            else:
                color = (0.7, 0.7, 0.7)

            conf = float(row["prediction_confidence"])
            conf = max(0, min(1, conf))

            # Border thickness
            thickness = 0.2 + 1.8 * conf
            page.draw_rect(bbox, color=color, width=thickness)

            # Add small label
            label_text = f"{conf:.2f}"
            text_pos = fitz.Point(row["x1"] - 20, row["y0"] + 2)

            page.insert_text(
                text_pos,
                label_text,
                fontsize=6,
                color=(0, 0, 0),         # black text
                fill_opacity=0.8
            )

    doc.save(output_path)
    doc.close()
    print("Saved:", output_path)

render_predictions_with_labels(new_pdf_file_path, df_romansh, "sursilvan_romansh_with_predictions.pdf")

Saved: sursilvan_romansh_with_predictions.pdf


In [364]:
df_romansh.to_csv("sursilvan_romansh_line_roles_predictions.csv", index=False)

In [365]:
# Load the active learning dataset
import json
corrected_labels_file = "sursilvan_romansh_corrected_labels.json"

with open(corrected_labels_file, "r", encoding="utf-8") as f:
    corrected_data = json.load(f)

corrected_data

FileNotFoundError: [Errno 2] No such file or directory: 'sursilvan_romansh_corrected_labels.json'

# Test 2

In [3]:

import pymupdf
import json

with open("kalamang_combined_final_roles.json","r",encoding="utf-8") as f:
    combined_final_roles = json.load(f)

In [4]:
pdf_file_path = "grammar_books/kalamang.pdf"

In [5]:
print("Opening PDF file...")
new_doc = pymupdf.open(pdf_file_path)
print("PDF file opened successfully.")

Opening PDF file...
PDF file opened successfully.


In [6]:
import json
import pandas as pd

def load_and_flatten(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        rows = json.load(f)

    data = []
    for row in rows:
        b = row["bline"]
        data.append({
            "page": row["page_num"],
            "line_no": b["line_no"],
            "text": b["text"],
            "x0": b["bbox"][0],
            "y0": b["bbox"][1],
            "x1": b["bbox"][2],
            "y1": b["bbox"][3],
            "width": b["bbox"][2] - b["bbox"][0],
            "height": b["bbox"][3] - b["bbox"][1],
            "role": row["role"]
        })
         

    return data

curated_data = load_and_flatten("kalamang_combined_final_roles.json")
curated_lookup = {}
for row in curated_data:

    curated_lookup[(row["page"], row["text"])] = row["role"]



In [11]:
# Create line level data to train model
all_merged_lines = []
for page in new_doc:
    page_num = page.number + 1
    
    merged_lines = extract_visual_lines(page)
    if merged_lines is None:
        continue
    all_merged_lines.append({
        "page_num": page_num,
        "lines": merged_lines
    })

# prepare data for prediction
all_data = []
for row in all_merged_lines:
    lines = row["lines"]
    for line_no,line in enumerate(lines):
        all_data.append({
            "page": row["page_num"],
            "line_no": line_no,
            "text": line["text"],
            "x0": line["x0"],
            "y0": line["y0"],
            "x1": line["x1"],
            "y1": line["y1"],
            "width": line["x1"] - line["x0"],
            "height": line["y1"] - line["y0"],
            "role": "OTHER"  # placeholder
        })
df = pd.DataFrame(all_data)


In [12]:
df.head()

,page,line_no,text,x0,y0,x1,y1,width,height,role
0,1,0,A grammar of,46.354000,50.835377,351.912201,100.382675,305.558201,49.547298,OTHER
1,1,1,Kalamang,46.354000,99.733368,269.415955,149.280670,223.061954,49.547302,OTHER
2,1,2,Eline Visser,46.354000,183.008514,169.292999,211.402039,122.938999,28.393524,OTHER
3,1,3,Comprehensive Grammar Library 4 language science,46.354000,592.548401,431.578400,606.221252,385.224400,13.672852,OTHER
4,1,4,press,403.896912,606.764160,423.339966,614.754333,19.443054,7.990173,OTHER


In [13]:
text_only_df = df[["text","role","page","line_no"]].copy()
text_only_df.head()

,text,role,page,line_no
0,A grammar of,OTHER,1,0
1,Kalamang,OTHER,1,1
2,Eline Visser,OTHER,1,2
3,Comprehensive Grammar Library 4 language science,OTHER,1,3
4,press,OTHER,1,4


In [14]:
text_only_df.to_csv("kalamang_text_only_data_for_bert_training.csv",index=False)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import re

In [2]:


MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128

# ---------------------------------------------------------
# Load your CSV
# ---------------------------------------------------------
df = pd.read_csv("kalamang_text_only_data_for_bert_training.csv").fillna("")
print(df.columns)
df = df.sort_values(["page", "line_no"])

# role is your label column
labels = sorted(df["role"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
df["label_id"] = df["role"].map(label2id)

# ---------------------------------------------------------
# Add simple context (prev / next lines)
# ---------------------------------------------------------
prev_lines = []
next_lines = []

for idx in range(len(df)):
    if idx > 0 and df.iloc[idx-1]["page"] == df.iloc[idx]["page"]:
        prev_lines.append(df.iloc[idx-1]["text"])
    else:
        prev_lines.append("")
    if idx < len(df)-1 and df.iloc[idx+1]["page"] == df.iloc[idx]["page"]:
        next_lines.append(df.iloc[idx+1]["text"])
    else:
        next_lines.append("")

df["prev"] = prev_lines
df["next"] = next_lines

df["context_text"] = (
    "[PREV] " + df["prev"] +
    " [CURR] " + df["text"] +
    " [NEXT] " + df["next"]
)



Index(['text', 'role', 'page', 'line_no'], dtype='object')


In [7]:
# ---------------------------------------------------------
# Structural feature extractor (safe across docs)
# ---------------------------------------------------------
def structural_features(line):
    tokens = line.split()
    ascii_chars = sum(c.isascii() for c in line)
    non_ascii = len(line) - ascii_chars

    return {
        "len": len(line),
        "num_tokens": len(tokens),
        "ascii_ratio": ascii_chars / max(1, len(line)),
        "non_ascii": non_ascii,
        "digit_ratio": sum(c.isdigit() for c in line) / max(1, len(line)),
        "has_gloss_markers": int(bool(re.search(r"[=~\-]", line))),
        "token_length_mean": np.mean([len(t) for t in tokens]) if tokens else 0,
        "token_length_std": np.std([len(t) for t in tokens]) if tokens else 0,
    }

feat_list = df["text"].apply(structural_features).tolist()
feat_df = pd.DataFrame(feat_list)
df = pd.concat([df, feat_df], axis=1)

# ---------------------------------------------------------
# Convert to HuggingFace dataset
# ---------------------------------------------------------
dataset = Dataset.from_pandas(df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tokenizer(
        batch["context_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    # attach structural features
    for key in ["len", "num_tokens", "ascii_ratio", "non_ascii",
                "digit_ratio", "has_gloss_markers",
                "token_length_mean", "token_length_std"]:
        enc[key] = batch[key]

    return enc

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label_id", "labels")

# Keep only needed tensors
cols = ["input_ids", "attention_mask", "labels",
        "len", "num_tokens", "ascii_ratio", "non_ascii",
        "digit_ratio", "has_gloss_markers", "token_length_mean", "token_length_std"]

dataset.set_format(type="torch", columns=cols)

# ---------------------------------------------------------
# Train/Validation split
# ---------------------------------------------------------
ds = dataset.train_test_split(test_size=0.1)


Map:   0%|          | 0/19096 [00:00<?, ? examples/s]

In [8]:

# ---------------------------------------------------------
# Model: BERT + structural features → MLP
# ---------------------------------------------------------
class LineClassifier(nn.Module):
    def __init__(self, bert_name, num_labels, feat_dim):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        hidden = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden + feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask,
                len, num_tokens, ascii_ratio, non_ascii,
                digit_ratio, has_gloss_markers,
                token_length_mean, token_length_std,
                labels=None):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output

        feats = torch.stack([
            len, num_tokens, ascii_ratio, non_ascii, digit_ratio,
            has_gloss_markers.float(), token_length_mean, token_length_std
        ], dim=1)

        combined = torch.cat([pooled, feats], dim=1)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}



In [9]:
torch.cuda.empty_cache()

In [10]:
feature_dim = 8
model = LineClassifier(MODEL_NAME, num_labels=len(labels), feat_dim=feature_dim)

# ---------------------------------------------------------
# Trainer
# ---------------------------------------------------------
training_args = TrainingArguments(
    output_dir="stage1_line_classifier",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
model = model.to(device)

# Move dataset to device


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model("stage1_line_classifier_final")


Using device: cuda


C:\Users\Varun\AppData\Local\Temp\ipykernel_33728\3265261300.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.000000
1000,0.000000
1500,0.000000
2000,0.000000
2500,0.000000
3000,0.000000
3500,0.000000
4000,0.000000
